## TransForm

In [1]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda

In [2]:
ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 231kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 4.30MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 23.3MB/s]


In [3]:
target_transform = Lambda(lambda y : torch.zeros(10, dtype=torch.float).scatter_(dim=0,index=torch.tensor(y),value=1))

# Build NN

In [4]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [6]:
print(f"{device}")

cpu


In [9]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [10]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [12]:
X = torch.rand(1,28,28,device = device)
logits = model(X)
pred_probab = nn.Softmax(dim = 1)(logits)
y_pred = pred_probab.argmax(1)
print({y_pred})

{tensor([8])}


In [13]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [14]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [19]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [20]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1462,  0.3656, -0.2454, -0.2638, -0.3958,  0.2296,  0.4058, -0.3443,
          0.2029, -0.5762,  0.0667,  0.2881,  0.5931, -0.1067,  0.0699,  0.4767,
          0.3997,  0.2864, -0.6505, -0.2755],
        [-0.1050,  0.2950, -0.3687, -0.1667, -0.3494,  0.0674,  0.2933,  0.2220,
          0.1033, -0.3179,  0.1279,  0.1357,  0.5978,  0.0100, -0.2469,  0.1716,
          0.3424,  0.0279, -0.1486, -0.1611],
        [ 0.2518,  0.3563, -0.3256, -0.3135, -0.4956,  0.2152,  0.0052, -0.2201,
          0.1899, -0.2020,  0.2718, -0.0548,  0.4578, -0.2653,  0.1828,  0.4583,
          0.3075,  0.4346, -0.4440, -0.3361]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.3656, 0.0000, 0.0000, 0.0000, 0.2296, 0.4058, 0.0000, 0.2029,
         0.0000, 0.0667, 0.2881, 0.5931, 0.0000, 0.0699, 0.4767, 0.3997, 0.2864,
         0.0000, 0.0000],
        [0.0000, 0.2950, 0.0000, 0.0000, 0.0000, 0.0674, 0.2933, 0.2220, 0.1033,
         0.0000, 0.1279, 0.1357, 0.5978, 0.0100, 0.00

In [21]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20,10)
)

input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [22]:
softmax = nn.Softmax(dim =1)
pred_probab = softmax(logits)

In [24]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0173, -0.0171,  0.0013,  ..., -0.0213, -0.0070, -0.0322],
        [ 0.0031, -0.0251, -0.0250,  ..., -0.0106, -0.0065,  0.0249]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0194, 0.0015], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0094, -0.0310,  0.0007,  ..., -0.0319,  0.0384,  0.0053],
        [-0.0015,  0.0188,  0.0013,  ...,  0.0413, -0.0347,  0.0293]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si